# CLS token embeddings

Reload the patch-4, 2D-position, dropout-0.1, step-LR class-token model and compare three vectors taken from the final token sequence of each test image:

- the class token
- the mean of the patch tokens (every token except the class token)
- the first patch token (the top-left patch)

Each vector is reduced from 384 dimensions to 2 with UMAP and drawn in its own panel. A separate figure shows the class token alone. Color is the original class, and the legend lists the class index and name. Silhouette and clustering scores are computed on the 384-dimensional vectors after the plots.


In [1]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import plotly.express as px
from plotly.subplots import make_subplots
import torch
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import umap

project_root = Path.cwd().resolve().parent
sys.path.insert(0, str(project_root))

from models.vit import VisionTransformer


/bigdata/md5_t1fada40cc125fa5312a398ad7cb/venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
CLASS_NAMES = [
    "airplane",
    "automobile",
    "bird",
    "cat",
    "deer",
    "dog",
    "frog",
    "horse",
    "ship",
    "truck",
]

# Same order as Plotly's default qualitative palette, one color per class.
CLASS_COLORS = px.colors.qualitative.Plotly[: len(CLASS_NAMES)]

checkpoint_path = (
    project_root
    / "runs"
    / "cls_patch4_pos2d_steplr25"
    / "vit_cls_patch4_pos2d_steplr25.pt"
)
checkpoint = torch.load(checkpoint_path, map_location="cpu", weights_only=False)
config = checkpoint["config"]
print("Loaded", checkpoint_path)
print(config)


Loaded /bigdata/md5_t1fada40cc125fa5312a398ad7cb/experiments/vit-cifar10/runs/cls_patch4_pos2d_steplr25/vit_cls_patch4_pos2d_steplr25.pt
{'image_size': 32, 'patch_size': 4, 'in_channels': 3, 'emb_size': 384, 'num_attention_heads': 6, 'num_transformer_blocks': 8, 'dropout': 0.1, 'n_classes': 10, 'pool': 'cls', 'pos_embed': '2d', 'lr': 0.0003, 'lr_step_epochs': 25, 'lr_gamma': 0.1}


In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = VisionTransformer(
    image_size=config["image_size"],
    patch_size=config["patch_size"],
    in_channels=config["in_channels"],
    emb_size=config["emb_size"],
    num_attention_heads=config["num_attention_heads"],
    dropout=config["dropout"],
    num_transformer_blocks=config["num_transformer_blocks"],
    n_classes=config["n_classes"],
    pool=config["pool"],
    pos_embed=config["pos_embed"],
).to(device)

model.load_state_dict(checkpoint["model_state_dict"])
model.eval()
print("Loaded weights on:", device)


Loaded weights on: cuda


In [4]:
eval_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(
        mean=(0.4914, 0.4822, 0.4465),
        std=(0.2470, 0.2435, 0.2616),
    ),
])

test_dataset = datasets.CIFAR10(
    root="../../../data",
    train=False,
    download=False,
    transform=eval_transform,
)

test_loader = DataLoader(
    test_dataset,
    batch_size=256,
    shuffle=False,
    num_workers=4,
    pin_memory=True,
)


`model.encode_tokens` returns every token after the transformer blocks and the final layer norm, shape `(batch, num_patches + 1, emb_size)`. Index 0 is the class token. The classification report below uses the classification head on that class token. Dropout is off because the model is in eval mode.


In [5]:
from sklearn.metrics import classification_report


@torch.no_grad()
def collect_embeddings(model, loader, device):
    cls_tokens = []
    mean_patches = []
    first_patches = []
    labels = []
    predictions = []

    for images, batch_labels in loader:
        images = images.to(device, non_blocking=True)
        tokens = model.encode_tokens(images)
        cls_token = tokens[:, 0, :]
        patch_tokens = tokens[:, 1:, :]

        cls_tokens.append(cls_token.float().cpu())
        mean_patches.append(patch_tokens.mean(dim=1).float().cpu())
        first_patches.append(patch_tokens[:, 0, :].float().cpu())
        labels.append(batch_labels)
        predictions.append(model.classifier(cls_token).argmax(dim=1).cpu())

    embeddings = {
        "CLS token": torch.cat(cls_tokens).numpy(),
        "Mean of patch tokens": torch.cat(mean_patches).numpy(),
        "First patch": torch.cat(first_patches).numpy(),
    }
    return embeddings, torch.cat(labels).numpy(), torch.cat(predictions).numpy()


embeddings, labels, predictions = collect_embeddings(model, test_loader, device)
for name, vectors in embeddings.items():
    print(name, vectors.shape)
print(
    classification_report(
        labels,
        predictions,
        labels=list(range(len(CLASS_NAMES))),
        target_names=CLASS_NAMES,
        digits=4,
    )
)


CLS token (10000, 384)
Mean of patch tokens (10000, 384)
First patch (10000, 384)
              precision    recall  f1-score   support

    airplane     0.7943    0.8340    0.8137      1000
  automobile     0.8702    0.9120    0.8906      1000
        bird     0.7676    0.7530    0.7602      1000
         cat     0.6721    0.6230    0.6466      1000
        deer     0.8019    0.7730    0.7872      1000
         dog     0.7218    0.7290    0.7254      1000
        frog     0.8456    0.8490    0.8473      1000
       horse     0.8250    0.8530    0.8387      1000
        ship     0.8775    0.8670    0.8722      1000
       truck     0.8652    0.8600    0.8626      1000

    accuracy                         0.8053     10000
   macro avg     0.8041    0.8053    0.8045     10000
weighted avg     0.8041    0.8053    0.8045     10000



Each panel is an independent UMAP of one vector per test image. The three reductions are not on a shared scale.


In [6]:
reducer_kwargs = dict(
    n_components=2,
    n_neighbors=15,
    min_dist=0.1,
    metric="euclidean",
    random_state=42,
)

coords = {}
for name, vectors in embeddings.items():
    coords[name] = umap.UMAP(**reducer_kwargs).fit_transform(vectors)
    print(name, coords[name].shape)


/bigdata/md5_t1fada40cc125fa5312a398ad7cb/venv/lib/python3.11/site-packages/umap/umap_.py:4160: FutureWarning: compatibility_layout was not set, so UMAP is using the legacy (0.5.x) nearest-neighbor, initialization and layout pipeline. In a future release the default will change to compatibility_layout=False, which uses recursive initialization and the Adam optimizer and produces different embeddings. Set compatibility_layout=True to keep the current behavior and silence this warning, or compatibility_layout=False (or optimizer='adam') to adopt the new behavior now.
  self.fit(X, y, ensure_all_finite, **kwargs)
/bigdata/md5_t1fada40cc125fa5312a398ad7cb/venv/lib/python3.11/site-packages/umap/umap_.py:3099: UserWarning: n_jobs value -1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


CLS token (10000, 2)


/bigdata/md5_t1fada40cc125fa5312a398ad7cb/venv/lib/python3.11/site-packages/umap/umap_.py:4160: FutureWarning: compatibility_layout was not set, so UMAP is using the legacy (0.5.x) nearest-neighbor, initialization and layout pipeline. In a future release the default will change to compatibility_layout=False, which uses recursive initialization and the Adam optimizer and produces different embeddings. Set compatibility_layout=True to keep the current behavior and silence this warning, or compatibility_layout=False (or optimizer='adam') to adopt the new behavior now.
  self.fit(X, y, ensure_all_finite, **kwargs)
/bigdata/md5_t1fada40cc125fa5312a398ad7cb/venv/lib/python3.11/site-packages/umap/umap_.py:3099: UserWarning: n_jobs value -1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


Mean of patch tokens (10000, 2)


/bigdata/md5_t1fada40cc125fa5312a398ad7cb/venv/lib/python3.11/site-packages/umap/umap_.py:4160: FutureWarning: compatibility_layout was not set, so UMAP is using the legacy (0.5.x) nearest-neighbor, initialization and layout pipeline. In a future release the default will change to compatibility_layout=False, which uses recursive initialization and the Adam optimizer and produces different embeddings. Set compatibility_layout=True to keep the current behavior and silence this warning, or compatibility_layout=False (or optimizer='adam') to adopt the new behavior now.
  self.fit(X, y, ensure_all_finite, **kwargs)
/bigdata/md5_t1fada40cc125fa5312a398ad7cb/venv/lib/python3.11/site-packages/umap/umap_.py:3099: UserWarning: n_jobs value -1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


First patch (10000, 2)


In [7]:
class_name = [f"{int(label)} {CLASS_NAMES[int(label)]}" for label in labels]
class_order = [f"{i} {name}" for i, name in enumerate(CLASS_NAMES)]
panel_names = list(embeddings)

fig = make_subplots(
    rows=1,
    cols=3,
    subplot_titles=panel_names,
    horizontal_spacing=0.04,
)

for col, name in enumerate(panel_names, start=1):
    frame = pd.DataFrame({
        "umap_1": coords[name][:, 0],
        "umap_2": coords[name][:, 1],
        "label": labels.astype(int),
        "class_name": class_name,
    })
    panel = px.scatter(
        frame,
        x="umap_1",
        y="umap_2",
        color="class_name",
        category_orders={"class_name": class_order},
        color_discrete_sequence=CLASS_COLORS,
        hover_data={"label": True, "class_name": True, "umap_1": ":.2f", "umap_2": ":.2f"},
        labels={"umap_1": "UMAP 1", "umap_2": "UMAP 2", "class_name": "Class"},
    )
    for trace in panel.data:
        trace.legendgroup = trace.name
        trace.showlegend = col == 1
        fig.add_trace(trace, row=1, col=col)

fig.update_traces(marker=dict(size=4, opacity=0.75))
fig.update_layout(
    title=dict(text="CIFAR-10 test set", x=0, xanchor="left"),
    legend=dict(
        title="Label and name",
        orientation="h",
        yanchor="top",
        y=-0.18,
        x=0,
    ),
    width=1400,
    height=560,
    margin=dict(t=90, b=120),
    updatemenus=[
        dict(
            type="buttons",
            direction="right",
            x=1,
            xanchor="right",
            y=1.16,
            yanchor="top",
            buttons=[
                dict(
                    label="Deselect all",
                    method="restyle",
                    args=[{"visible": "legendonly"}],
                ),
                dict(
                    label="Select all",
                    method="restyle",
                    args=[{"visible": True}],
                ),
            ],
        )
    ],
)
fig.show()


The class token on its own. Same UMAP coordinates as the first panel above.

In [8]:
cls_frame = pd.DataFrame({
    "umap_1": coords["CLS token"][:, 0],
    "umap_2": coords["CLS token"][:, 1],
    "label": labels.astype(int),
    "class_name": class_name,
})
cls_fig = px.scatter(
    cls_frame,
    x="umap_1",
    y="umap_2",
    color="class_name",
    category_orders={"class_name": class_order},
    color_discrete_sequence=CLASS_COLORS,
    hover_data={"label": True, "class_name": True, "umap_1": ":.2f", "umap_2": ":.2f"},
    labels={"umap_1": "UMAP 1", "umap_2": "UMAP 2", "class_name": "Class"},
    title="CLS token",
)
cls_fig.update_traces(marker=dict(size=6, opacity=0.75))
cls_fig.update_layout(
    legend=dict(title="Label and name", orientation="v"),
    width=860,
    height=700,
)
cls_fig.show()


## Class token from the three trained readouts

The patch-4, 2D, dropout-0.1, step-LR setting was trained three times. One model reads the class token, one reads the mean of the patch tokens, and one reads the top-left patch. Each model still carries a class token. The panels below are that class token, one independent UMAP per trained model. The first panel is the same reduction as the class-token figure above.

In [9]:
def load_readout(run_name):
    path = project_root / "runs" / run_name / f"vit_{run_name}.pt"
    saved = torch.load(path, map_location="cpu", weights_only=False)
    cfg = saved["config"]
    net = VisionTransformer(
        image_size=cfg["image_size"],
        patch_size=cfg["patch_size"],
        in_channels=cfg["in_channels"],
        emb_size=cfg["emb_size"],
        num_attention_heads=cfg["num_attention_heads"],
        dropout=cfg["dropout"],
        num_transformer_blocks=cfg["num_transformer_blocks"],
        n_classes=cfg["n_classes"],
        pool=cfg["pool"],
        pos_embed=cfg["pos_embed"],
    ).to(device)
    net.load_state_dict(saved["model_state_dict"])
    net.eval()
    print("Loaded", path)
    return net


@torch.no_grad()
def collect_cls(net, loader, device):
    pieces = []
    for images, _batch_labels in loader:
        images = images.to(device, non_blocking=True)
        pieces.append(net.encode_tokens(images)[:, 0, :].float().cpu())
    return torch.cat(pieces).numpy()


readout_cls = {
    "CLS model": embeddings["CLS token"],
    "Mean model": collect_cls(load_readout("mean_patch4_pos2d_steplr25"), test_loader, device),
    "First-patch model": collect_cls(load_readout("patch0_patch4_pos2d_steplr25"), test_loader, device),
}
for name, vectors in readout_cls.items():
    print(name, vectors.shape)


Loaded /bigdata/md5_t1fada40cc125fa5312a398ad7cb/experiments/vit-cifar10/runs/mean_patch4_pos2d_steplr25/vit_mean_patch4_pos2d_steplr25.pt


Loaded /bigdata/md5_t1fada40cc125fa5312a398ad7cb/experiments/vit-cifar10/runs/patch0_patch4_pos2d_steplr25/vit_patch0_patch4_pos2d_steplr25.pt


CLS model (10000, 384)
Mean model (10000, 384)
First-patch model (10000, 384)


In [10]:
readout_coords = {"CLS model": coords["CLS token"]}
for name, vectors in readout_cls.items():
    if name == "CLS model":
        continue
    readout_coords[name] = umap.UMAP(**reducer_kwargs).fit_transform(vectors)
    print(name, readout_coords[name].shape)


/bigdata/md5_t1fada40cc125fa5312a398ad7cb/venv/lib/python3.11/site-packages/umap/umap_.py:4160: FutureWarning: compatibility_layout was not set, so UMAP is using the legacy (0.5.x) nearest-neighbor, initialization and layout pipeline. In a future release the default will change to compatibility_layout=False, which uses recursive initialization and the Adam optimizer and produces different embeddings. Set compatibility_layout=True to keep the current behavior and silence this warning, or compatibility_layout=False (or optimizer='adam') to adopt the new behavior now.
  self.fit(X, y, ensure_all_finite, **kwargs)
/bigdata/md5_t1fada40cc125fa5312a398ad7cb/venv/lib/python3.11/site-packages/umap/umap_.py:3099: UserWarning: n_jobs value -1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


Mean model (10000, 2)


/bigdata/md5_t1fada40cc125fa5312a398ad7cb/venv/lib/python3.11/site-packages/umap/umap_.py:4160: FutureWarning: compatibility_layout was not set, so UMAP is using the legacy (0.5.x) nearest-neighbor, initialization and layout pipeline. In a future release the default will change to compatibility_layout=False, which uses recursive initialization and the Adam optimizer and produces different embeddings. Set compatibility_layout=True to keep the current behavior and silence this warning, or compatibility_layout=False (or optimizer='adam') to adopt the new behavior now.
  self.fit(X, y, ensure_all_finite, **kwargs)
/bigdata/md5_t1fada40cc125fa5312a398ad7cb/venv/lib/python3.11/site-packages/umap/umap_.py:3099: UserWarning: n_jobs value -1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


First-patch model (10000, 2)


In [11]:
readout_names = list(readout_cls)
readout_fig = make_subplots(
    rows=1,
    cols=3,
    subplot_titles=readout_names,
    horizontal_spacing=0.04,
)

for col, name in enumerate(readout_names, start=1):
    frame = pd.DataFrame({
        "umap_1": readout_coords[name][:, 0],
        "umap_2": readout_coords[name][:, 1],
        "label": labels.astype(int),
        "class_name": class_name,
    })
    panel = px.scatter(
        frame,
        x="umap_1",
        y="umap_2",
        color="class_name",
        category_orders={"class_name": class_order},
        color_discrete_sequence=CLASS_COLORS,
        hover_data={"label": True, "class_name": True, "umap_1": ":.2f", "umap_2": ":.2f"},
        labels={"umap_1": "UMAP 1", "umap_2": "UMAP 2", "class_name": "Class"},
    )
    for trace in panel.data:
        trace.legendgroup = trace.name
        trace.showlegend = col == 1
        readout_fig.add_trace(trace, row=1, col=col)

readout_fig.update_traces(marker=dict(size=4, opacity=0.75))
readout_fig.update_layout(
    title=dict(text="Class token from each trained readout", x=0, xanchor="left"),
    legend=dict(title="Label and name", orientation="h", yanchor="top", y=-0.18, x=0),
    width=1400,
    height=560,
    margin=dict(t=90, b=120),
)
readout_fig.show()


These scores use the 384-dimensional vectors, not the UMAP coordinates. Silhouette treats the CIFAR-10 label as the cluster id: 1 means each class is compact and separated, 0 means the classes overlap, and negative means a point is closer to another class. Adjusted Rand and normalized mutual information compare a label-free 10-means clustering with the true labels: 0 is chance and 1 is a perfect match. Vectors are L2-normalized so Euclidean distance matches cosine distance. Every score uses all 10,000 test images.


In [12]:
from sklearn.cluster import KMeans
from sklearn.metrics import (
    adjusted_rand_score,
    normalized_mutual_info_score,
    silhouette_samples,
)
from sklearn.preprocessing import normalize

for name, vectors in embeddings.items():
    vectors_unit = normalize(vectors)
    silhouette = silhouette_samples(vectors_unit, labels, metric="euclidean")
    print(f"\n{name}")
    print(f"Silhouette (true labels, all {len(labels):,} images): {silhouette.mean():.4f}")
    for class_id, class_name in enumerate(CLASS_NAMES):
        class_score = silhouette[labels == class_id].mean()
        print(f"  {class_id} {class_name:<12} {class_score:.4f}")

    cluster_ids = KMeans(
        n_clusters=len(CLASS_NAMES),
        n_init=10,
        random_state=42,
    ).fit_predict(vectors_unit)
    print(f"Adjusted Rand (KMeans vs labels): {adjusted_rand_score(labels, cluster_ids):.4f}")
    print(f"NMI (KMeans vs labels): {normalized_mutual_info_score(labels, cluster_ids):.4f}")

    cluster_ids = KMeans(
        n_clusters=len(CLASS_NAMES),
        n_init=10,
        random_state=42,
    ).fit_predict(vectors_unit)
    print(f"Adjusted Rand (KMeans vs labels): {adjusted_rand_score(labels, cluster_ids):.4f}")
    print(f"NMI (KMeans vs labels): {normalized_mutual_info_score(labels, cluster_ids):.4f}")



CLS token
Silhouette (true labels, all 10,000 images): 0.1464
  0 airplane     0.1249
  1 automobile   0.2571
  2 bird         0.0598
  3 cat          -0.0031
  4 deer         0.0892
  5 dog          0.0752
  6 frog         0.1945
  7 horse        0.2102
  8 ship         0.2455
  9 truck        0.2108


Adjusted Rand (KMeans vs labels): 0.6104
NMI (KMeans vs labels): 0.6434


Adjusted Rand (KMeans vs labels): 0.6104
NMI (KMeans vs labels): 0.6434



Mean of patch tokens
Silhouette (true labels, all 10,000 images): 0.0751
  0 airplane     0.0498
  1 automobile   0.1619
  2 bird         -0.0183
  3 cat          -0.0150
  4 deer         0.0497
  5 dog          0.0296
  6 frog         0.1615
  7 horse        0.0848
  8 ship         0.1265
  9 truck        0.1207


Adjusted Rand (KMeans vs labels): 0.4673
NMI (KMeans vs labels): 0.5327


Adjusted Rand (KMeans vs labels): 0.4673
NMI (KMeans vs labels): 0.5327



First patch
Silhouette (true labels, all 10,000 images): 0.0085
  0 airplane     0.0166
  1 automobile   0.0619
  2 bird         -0.0551
  3 cat          -0.0381
  4 deer         0.0214
  5 dog          -0.0222
  6 frog         0.0960
  7 horse        -0.0056
  8 ship         0.0237
  9 truck        -0.0138


Adjusted Rand (KMeans vs labels): 0.2082
NMI (KMeans vs labels): 0.3067


Adjusted Rand (KMeans vs labels): 0.2082
NMI (KMeans vs labels): 0.3067


The plot looks clustered, but the picture is not the measurement. UMAP only draws the 384-dimensional vectors in 2D. The three scores were computed on those 384-d vectors after L2 normalization, on all 10,000 official test images. Each table is one trained model: patch 4, 2D positions, dropout 0.1, step LR. The same three vectors are scored every time. The bold row is the vector the classification head was trained on. The first patch is the top-left image patch, not the class token.

**Trained on the class token** (`cls_patch4_pos2d_steplr25`). The head's test accuracy is 80.53%.

| Vector | Silhouette | Adjusted Rand | NMI |
|---|---:|---:|---:|
| **Class token** | **0.15** | **0.61** | **0.64** |
| Mean of patch tokens | 0.08 | 0.47 | 0.53 |
| First patch | 0.01 | 0.21 | 0.31 |

**Trained on the mean** (`mean_patch4_pos2d_steplr25`). The head's test accuracy is 82.11%.

| Vector | Silhouette | Adjusted Rand | NMI |
|---|---:|---:|---:|
| Class token | 0.07 | 0.41 | 0.52 |
| **Mean of patch tokens** | **0.19** | **0.64** | **0.66** |
| First patch | 0.08 | 0.43 | 0.47 |

**Trained on the first patch** (`patch0_patch4_pos2d_steplr25`). The head's test accuracy is 81.10%.

| Vector | Silhouette | Adjusted Rand | NMI |
|---|---:|---:|---:|
| Class token | 0.18 | 0.57 | 0.63 |
| Mean of patch tokens | 0.19 | 0.58 | 0.63 |
| **First patch** | **0.18** | **0.60** | **0.65** |
